# Demo

In [1]:
import json
import re

import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import tokenizer_from_json


INPUT_DATA = "./data/test.csv"
OUTPUT_FILE = "Group_57_B.csv"

MODEL_PATH = "Group_57_B_model.keras"
TOKENIZER_PATH = "Group_57_B_tokenizer.json"
CONFIG_PATH = "Group_57_B_config.json"

/Users/danielkoshovyy/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


#### A custom Keras layer used in the model architecture.
Need this since doing it with lambda functions does not allow keras to save the model.

In [2]:
@tf.keras.utils.register_keras_serializable()
class AbsoluteDifference(tf.keras.layers.Layer):
    def call(self, inputs):
        x1, x2 = inputs
        return tf.abs(x1 - x2)

    def get_config(self):
        return super().get_config()

## Helper Functions

In [3]:
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def fix_empty_sequences(sequences, oov_index):
    fixed = []

    for seq in sequences:
        if len(seq) == 0:
            fixed.append([oov_index])
        else:
            fixed.append(seq)

    return fixed

## Load Model, Tokenizer and Config
##### Loads the trained BiLSTM model, the tokenizer used during training,
##### and the config file containing max lengths and the best decision threshold.

In [4]:
# load model, tokenizer, and config
model = tf.keras.models.load_model(MODEL_PATH)

with open(TOKENIZER_PATH) as f:
    tokenizer = tokenizer_from_json(f.read())

with open(CONFIG_PATH) as f:
    config = json.load(f)

# extract config values
max_premise_length = config["max_premise_length"]
max_hypothesis_length = config["max_hypothesis_length"]
best_threshold = config["best_threshold"]
oov_token = config["oov_token"]

## Load and Preprocess Test Data

In [ ]:
# load test data
data_df = pd.read_csv(INPUT_DATA)

# clean text columns
data_df["premise"] = data_df["premise"].fillna("").apply(clean_text)
data_df["hypothesis"] = data_df["hypothesis"].fillna("").apply(clean_text)

premises = data_df["premise"].tolist()
hypotheses = data_df["hypothesis"].tolist()

# tokenize
premise_seq = tokenizer.texts_to_sequences(premises)
hypothesis_seq = tokenizer.texts_to_sequences(hypotheses)

# get OOV index from tokenizer
oov_index = tokenizer.word_index[oov_token]

premise_seq = fix_empty_sequences(premise_seq, oov_index)
hypothesis_seq = fix_empty_sequences(hypothesis_seq, oov_index)


# pad sequences
premise_array = pad_sequences(
    premise_seq,
    maxlen=max_premise_length,
    padding="post",
    truncating="post"
)

hypothesis_array = pad_sequences(
    hypothesis_seq,
    maxlen=max_hypothesis_length,
    padding="post",
    truncating="post"
)


## Generate and Predictions

In [6]:
probs = model.predict(
    {
        "premise_input": premise_array,
        "hypothesis_input": hypothesis_array
    },
    verbose=0
).reshape(-1)

preds = (probs >= best_threshold).astype(int)


# save predictions
pd.DataFrame({"label": preds}).to_csv(OUTPUT_FILE, index=False)

print(f"saved predictions to {OUTPUT_FILE}")
print(f"rows: {len(preds)}")
print(f"threshold used: {best_threshold:.2f}")

saved predictions to Group_57_B.csv
rows: 3302
threshold used: 0.42
